In [ ]:
import pickle
from scipy.ndimage import binary_dilation
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Load in your pickle file
with open("results/chesapeake_bay_large_v2_R00_results.pkl", "rb") as f:
    obj = pickle.load(f)

# Examine the pickle file
print(type(obj))

if isinstance(obj, dict):
    print(obj.keys())

In [ ]:
# Examine shape of data
idx = 0

x = obj["ins_testing"][idx]
y_true = obj["outs_testing"][idx]
y_pred_prob = obj["predict_testing"][idx]

# Convert probabilities -> predicted class
y_pred = np.argmax(y_pred_prob, axis=-1)

print(x.shape)
print(y_true.shape)
print(y_pred.shape)

In [ ]:
# Check the worst and best accuracies from training
pred = np.argmax(obj["predict_testing"], axis=-1)
truth = obj["outs_testing"]

pixel_acc = np.mean(pred == truth, axis=(1,2))

worst = np.argsort(pixel_acc)[:20]
best = np.argsort(pixel_acc)[-20:]

print("Worst accuracies:", pixel_acc[worst])
print("Best accuracies:", pixel_acc[best])

In [ ]:
# Grab number of images from testing set
num_images = len(truth)
print(f"Number of test images: {num_images}")

# Compute predictions for all test images
pred_all = np.argmax(obj["predict_testing"], axis=-1)
truth_all = obj["outs_testing"]

# Pixel accuracy for each patch
pixel_acc = np.mean(pred_all == truth_all, axis=(1, 2))

In [ ]:
# Adjust what examples you want to look at
indices = np.arange(50, 70)

class_names = {
    0: "Background",
    1: "Water",
    2: "Tree Canpoy/Forest",
    3: "Low Vegetation/Field",
    4: "Barren Land",
    5: "Impervious (other)",
    6: "Impervious (road)"
}

for idx in indices:

    rgb = obj["ins_testing"][idx][:, :, :3].astype(float)

    truth = obj["outs_testing"][idx]
    pred = np.argmax(obj["predict_testing"][idx], axis=-1)
    error = pred != truth

    ##
    # Code for error outlines (not useful but interesting to look at)
    
    # # Create a 1-pixel outline around the error regions
    # outline = binary_dilation(error) ^ error
    
    # # Copy the image and draw the outline in red
    # overlay = rgb.copy()
    # overlay[outline] = [1.0, 0.0, 0.0]

    fig, ax = plt.subplots(1, 4, figsize=(15, 5))

    ax[0].imshow(rgb)
    ax[0].set_title(f"Input (#{idx})")

    ax[1].imshow(truth, cmap="viridis", vmin=0, vmax=6)
    ax[1].set_title("Ground Truth")

    ax[2].imshow(pred, cmap="viridis", vmin=0, vmax=6)
    ax[2].set_title(f"Prediction\nAccuracy = {pixel_acc[idx]:.3f}")

    ax[3].imshow(rgb)
    ax[3].imshow(error, cmap="Reds", alpha=0.6)
    ax[3].set_title(f"Error overlay ({error.sum():,} pixels)")

    for a in ax:
        a.axis("off")

    cmap = plt.get_cmap("viridis", 7)

    # Create legend on this figure
    legend_elements = [
        Patch(
            facecolor=cmap(i),
            label=class_names[i]
        )
        for i in range(7)
    ]

    fig.legend(
        handles=legend_elements,
        title="Classes",
        loc="center right",
        bbox_to_anchor=(1.15, 0.5)
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# This overlays the predicted segemntation over the original input image
idx = 69

rgb = obj["ins_testing"][idx][:, :, :3].astype(float)
truth = obj["outs_testing"][idx]
pred = np.argmax(obj["predict_testing"][idx], axis=-1)
error = pred != truth

plt.figure(figsize=(8, 8))

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].imshow(rgb)
ax[0].set_title("RGB")
ax[0].axis("off")

ax[1].imshow(pred, cmap="viridis", vmin=0, vmax=6)
ax[1].set_title("Prediction")
ax[1].axis("off")

# Predicted segmentation overlay (the higher the alpha the more prevalent the segmentation image)
im = ax[2].imshow(confidence, cmap="viridis", alpha=0.4, vmin=0, vmax=1)
ax[2].set_title("Confidence")
ax[2].axis("off")

fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# Select your index here
idx = 10

rgb = obj["ins_testing"][idx][:, :, :3].astype(float)
truth = obj["outs_testing"][idx]

probs = obj["predict_testing"][idx]
pred = np.argmax(probs, axis=-1)
confidence = np.max(probs, axis=-1)

error = pred != truth

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# Original image
ax[0].imshow(rgb)
ax[0].set_title("Original")
ax[0].axis("off")

# Predicted segmentation overlay (adjust alpha to change opacity) 
ax[1].imshow(rgb)
ax[1].imshow(pred, cmap="viridis", alpha=0.4, vmin=0, vmax=6)
ax[1].set_title("Prediction")
ax[1].axis("off")

# Confidence overlay (adjust alpha to change opacity)
ax[2].imshow(rgb)
im = ax[2].imshow(confidence, cmap="viridis", alpha=0.9, vmin=0, vmax=1)
ax[2].set_title("Confidence Overlay")
ax[2].axis("off")

fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04, label="Confidence")

plt.tight_layout()
plt.show()